# Three-way kernel-latency comparison — QAT vs PTQ vs FP16

Rigorous, honest layer-by-layer comparison from the trtexec `--exportProfile` JSONs.
**Analysis only** — the three JSONs are read-only inputs; nothing is built.

Key rigor point: a kernel like `model.8...cv1.conv.weight + .../QuantizeLinear + .../Conv + PWN(Sigmoid, Mul)`
is a **fused conv+quant+SiLU** kernel — the QuantizeLinear is *folded into the conv*, so its cost is **not** separable.
Only standalone `__myl_MulMinMaxRoun...` kernels count as **pure Q/DQ latency**.


In [ ]:
import json, re
import pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 90)

KJ = Path('../Per_kernel_json_file')   # read-only inputs
FILES = {'QAT':'qat_batch32_per_kernel_profile.json',
         'PTQ':'ptq_int8_per_kernel_profile.json',
         'FP16':'fp16_per_kernel_profile.json'}
# whole-engine medians measured on an exclusive idle A100 (scorecard)
WHOLE_ENGINE_MED = {'QAT':1.200, 'PTQ':1.070, 'FP16':1.127}


## STEP 1 — load each JSON → dataframe (medianMs is the latency metric)

In [ ]:
def load_profile(path):
    """trtexec --exportProfile JSON -> df[name, averageMs, medianMs, percentage].
    Element [0] is a {"count": N} header — skipped."""
    raw = json.load(open(path))
    rows = [e for e in raw if 'name' in e]
    df = pd.DataFrame(rows)[['name','averageMs','medianMs','percentage']]
    for c in ['averageMs','medianMs','percentage']:
        df[c] = pd.to_numeric(df[c])
    return df

raw_df = {t: load_profile(KJ/FILES[t]) for t in FILES}
for t, df in raw_df.items():
    print(f'{t:5} kernels={len(df):4d}  median-sum={df["medianMs"].sum():.4f} ms  '
          f'(whole-engine measured {WHOLE_ENGINE_MED[t]} ms)')


## STEP 2 — parse each kernel name into structured fields
The honest part: `role` distinguishes **fused conv+SiLU (Q folded in)** from **pure standalone Q/DQ**.

In [ ]:
def parse_name(name):
    n = name; nl = name.lower()
    mg = re.search(r'model[./](\d+)', n)
    layer = f'model.{mg.group(1)}' if mg else ('NMS/foreign' if ('topk' in nl or '__myl' in nl) else 'other')
    sub = re.search(r'model[./]\d+[./]([\w./]+?)/(?:conv|weight|act|Conv|Split)', n)
    subpath = sub.group(1) if sub else ''
    has_conv = 'conv' in nl
    has_Q    = 'quantizelinear' in nl or 'minmaxroun' in nl
    has_SiLU = ('sigmoid' in nl and 'mul' in nl)
    has_PWN  = 'pwn(' in nl
    reformat = ('reformat' in nl or 'copynode' in nl)
    if reformat: role = 'reformat'
    elif 'topk' in nl: role = 'NMS/TopK'
    elif ('matmul' in nl or '/attn/' in nl or 'maxrsubexpsum' in nl): role = 'attention'
    elif has_conv and has_SiLU and has_PWN: role = 'conv+SiLU (fused, Q folded in)'
    elif has_conv: role = 'conv-only'
    elif nl.strip().startswith('pwn(') and has_SiLU: role = 'standalone-SiLU'
    elif has_Q and not has_conv: role = 'pure Q/DQ (standalone)'
    else: role = 'other'
    # QuantizeLinear folded INTO the conv here -> its cost is NOT separable
    q_folded_into_conv = has_conv and 'quantizelinear' in nl
    return dict(layer=layer, subpath=subpath, has_conv=has_conv, has_Q=has_Q,
                has_SiLU=has_SiLU, has_PWN=has_PWN, reformat=reformat,
                role=role, q_folded_into_conv=q_folded_into_conv)

# demo on a fused kernel — show it is NOT counted as pure Q/DQ
demo = raw_df['QAT'][raw_df['QAT']['name'].str.contains('QuantizeLinear.*Conv.*PWN', regex=True)].iloc[0]['name']
print('example fused kernel:\n ', demo[:110], '...')
print('parsed:', parse_name(demo))


In [ ]:
def build_df(tag):
    df = load_profile(KJ/FILES[tag])
    parsed = df['name'].apply(parse_name).apply(pd.Series)
    out = pd.concat([df, parsed], axis=1); out['engine'] = tag
    return out

DFS = {t: build_df(t) for t in FILES}
print('parsed:', {t: DFS[t].shape for t in DFS})


## STEP 3 — every kernel mapped to median latency, grouped by layer (per engine)

In [ ]:
def per_layer_table(tag):
    d = DFS[tag].copy()
    d['flags'] = d.apply(lambda r: '+'.join([k for k in ['has_conv','has_Q','has_SiLU','has_PWN','reformat']
                                             if r[k]]), axis=1)
    return d[['layer','name','medianMs','role','flags']].sort_values(['layer','medianMs'], ascending=[True,False])

tbl_qat = per_layer_table('QAT')
tbl_qat.head(25)   # change 'QAT' -> 'PTQ'/'FP16' to view the others; full tables saved as CSV below


## STEP 4 — common comparison set (match by LAYER + op-ROLE, not exact string)
Since fusion differs, we compare at the layer/role level.

In [ ]:
ROLES = ['conv+SiLU (fused, Q folded in)','conv-only','standalone-SiLU',
         'pure Q/DQ (standalone)','attention','reformat','NMS/TopK','other']
op_role = pd.DataFrame({t: DFS[t].groupby('role')['medianMs'].sum() for t in FILES}).reindex(ROLES).fillna(0)
op_role_n = pd.DataFrame({t: DFS[t].groupby('role').size() for t in FILES}).reindex(ROLES).fillna(0).astype(int)
print('summed median ms by op-role x engine:'); display(op_role.round(4))
print('kernel COUNT by op-role x engine:'); display(op_role_n)
# layers present in ALL three engines (fair set)
common_layers = set(DFS['QAT']['layer']) & set(DFS['PTQ']['layer']) & set(DFS['FP16']['layer'])
print('layers present in ALL 3 engines:', sorted(common_layers, key=lambda s:(s!='NMS/foreign', s)))


## STEP 5 — fair per-layer comparison (branching C2f layers) + grouped bar
Flag: at deep C3k2 blocks (model.6/8/13/19) FP16 also splits SiLU, so a direct compare there is
partly apples-to-oranges — the QAT-specific effect shows cleanest at model.2/4/16.

In [ ]:
BRANCH = [f'model.{i}' for i in [2,4,6,8,13,16,19]]
comp = pd.DataFrame({t: DFS[t][DFS[t]['layer'].isin(BRANCH)].groupby('layer')['medianMs'].sum()
                     for t in FILES}).reindex(BRANCH).fillna(0)
display(comp.round(4))

x = np.arange(len(comp)); w = 0.27; colors = {'QAT':'#c0392b','PTQ':'#2980b9','FP16':'#27ae60'}
fig, ax = plt.subplots(figsize=(10,4.5))
for i, t in enumerate(['QAT','PTQ','FP16']):
    ax.bar(x+(i-1)*w, comp[t], w, label=t, color=colors[t])
ax.set_xticks(x); ax.set_xticklabels(comp.index, rotation=30, ha='right')
ax.set_ylabel('summed median latency (ms)'); ax.set_title('Per-layer median latency — branching C2f layers')
ax.legend(); plt.tight_layout(); plt.savefig('layer_comparison_branching.png', dpi=130); plt.show()


## STEP 6 — per-engine op-category summary + combined grouped bar

In [ ]:
# one figure per engine
for t in FILES:
    s = op_role[t][op_role[t] > 0].sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8,3.5))
    ax.barh(s.index, s.values, color=colors[t])
    for i,v in enumerate(s.values): ax.text(v+0.003, i, f'{v:.3f}', va='center', fontsize=8)
    ax.set_xlabel('summed median ms'); ax.set_title(f'{t} — latency by op-category'); ax.set_xlim(0, op_role.values.max()*1.15)
    plt.tight_layout(); plt.savefig(f'opcat_{t}.png', dpi=130); plt.show()

# combined grouped bar
x = np.arange(len(op_role)); fig, ax = plt.subplots(figsize=(12,4.5))
for i, t in enumerate(['QAT','PTQ','FP16']):
    ax.bar(x+(i-1)*w, op_role[t], w, label=t, color=colors[t])
ax.set_xticks(x); ax.set_xticklabels(op_role.index, rotation=30, ha='right')
ax.set_ylabel('summed median ms'); ax.set_title('Latency by op-category — QAT vs PTQ vs FP16'); ax.legend()
plt.tight_layout(); plt.savefig('opcat_combined.png', dpi=130); plt.show()


## STEP 7 — sanity reconciliation
Per-kernel median sums are **inflated by CUDA-event instrumentation** — they don't equal the whole-engine median.

In [ ]:
recon = pd.DataFrame({
    'kernels':      {t: len(DFS[t]) for t in FILES},
    'median_sum_ms':{t: DFS[t]['medianMs'].sum() for t in FILES},
    'measured_ms':  WHOLE_ENGINE_MED,
})
recon['inflation_x'] = (recon['median_sum_ms']/recon['measured_ms']).round(2)
display(recon.round(4))
print('Per-kernel median sums are ~1.4-1.6x the true whole-engine median (instrumentation overhead).')
print('Use SHARES / relative comparisons across kernels; the absolute per-kernel ms are diagnostic, not deployable.')

# save CSVs
for t in FILES: DFS[t].to_csv(f'kernels_{t}.csv', index=False)
op_role.to_csv('op_role_by_engine.csv'); comp.to_csv('layer_comparison_branching.csv')
print('Saved CSVs: kernels_{QAT,PTQ,FP16}.csv, op_role_by_engine.csv, layer_comparison_branching.csv')
